# KIRC-Hetionet : Download and Subgraph

Gene ID standardization + a single switchable **Pathway / Biological Process** graph context.

Flow:

```text
 1. Environment                  7. Mapping validation
 2. Google Drive mount           8. Context configuration
 3. Project paths                9. Pathway / Biological Process pipeline
 4. Hetionet loading            10. Result comparison
 5. KIRC loading                11. README update
 6. Gene ID standardization     12. Google Drive backup
```

Two rules this notebook enforces:

* the graph join key is **`Gene::Entrez`**, never a gene symbol;
* Pathway and Biological Process are **one pipeline** selected by `GRAPH_CONTEXT`,
  never two code paths.

Not in this stage: variance scoring, candidate ranking, ML, GNN, biomarker selection.

## 1. Environment

In [ ]:
# Dependencies (Colab already ships pandas; this is a no-op there).
try:
    import pandas as pd
except ImportError:  # pragma: no cover
    !pip install -q pandas
    import pandas as pd

import sys, os
from pathlib import Path

print("python :", sys.version.split()[0])
print("pandas :", pd.__version__)

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print("colab  :", IN_COLAB)

## 2. Google Drive mount

In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/KIRC_Hetionet_Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("mounted ->", DRIVE_PROJECT_DIR)
else:
    print("[skip] not in Colab - Drive cannot be mounted here.")
    print("       Set KIRC_HETIONET_DRIVE_DIR to mirror the backup locally.")

## 3. Project paths

The project code lives in the repository. In Colab it is cloned into `/content`;
locally the notebook just walks up to the repository root.

In [ ]:
REPO_URL = "https://github.com/kwak-lazy/solid-revice.git"
BRANCH = "claude/determined-lovelace-mj90py"


def locate_project() -> Path:
    """Find the project root (the directory holding config.py)."""
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "config.py").exists() and (candidate / "src").exists():
            return candidate
    if IN_COLAB:
        target = Path("/content/solid-revice")
        if not (target / "config.py").exists():
            !git clone --branch {BRANCH} --depth 1 {REPO_URL} {target}
        return target
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from inside the "
        "repository, or set KIRC_HETIONET_PROJECT_DIR."
    )


PROJECT_DIR = locate_project()
os.environ.setdefault("KIRC_HETIONET_PROJECT_DIR", str(PROJECT_DIR))
for p in (str(PROJECT_DIR), str(PROJECT_DIR / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

import config
from kirc_hetionet import hetionet, kirc, gene_mapping, context, readme_log, drive as drive_sync

config.ensure_dirs()
print("project dir :", config.PROJECT_DIR)
print("results dir :", config.RESULTS_DIR)
print("drive dir   :", config.DRIVE_PROJECT_DIR)

## 4. Hetionet loading

Download -> load nodes / edges -> inspect node kinds -> inspect edge types.

The edge file is stored with **git-lfs**, so it is fetched from the media
endpoint and checksum-verified; the plain `raw.githubusercontent.com` URL
serves a 133-byte LFS pointer instead of the data.

In [ ]:
paths = hetionet.download_hetionet()
nodes = hetionet.load_nodes()
edges = hetionet.load_edges()

print("nodes:", nodes.shape, " edges:", edges.shape)
nodes.head()

In [ ]:
# node types
hetionet.node_kind_counts(nodes)

In [ ]:
# edge types (metaedges)
hetionet.metaedge_counts(edges)

In [ ]:
# Hetionet Gene nodes, with the Entrez id parsed out of "Gene::<entrez>"
het_genes = hetionet.gene_nodes(nodes)
print(f"Hetionet Gene nodes: {len(het_genes):,}")
het_genes.head()

## 5. KIRC loading

Existing artifacts are checked **before** anything is regenerated:

```text
kirc_gene_mapping_all.tsv
kirc_hetionet_gene_nodes.tsv
kirc_expression_standardized.tsv.gz
```

If the expression matrix is not reachable, the pipeline falls back to the gene
coverage established by a prior run and says so - it never pretends the
Ensembl leg ran.

In [ ]:
kirc_source = kirc.resolve_kirc_genes()
print("\nKIRC gene source :", kirc_source["source"])
print("path             :", kirc_source["path"])
if kirc_source["ensembl_ids"] is not None:
    print(f"KIRC Ensembl IDs : {len(kirc_source['ensembl_ids']):,}")
else:
    print(f"Hetionet gene ids: {len(kirc_source['hetionet_gene_ids'] or []):,}")

## 6. Gene ID standardization

```text
KIRC Ensembl Gene ID
        |  HGNC complete set (annotation)
        v
    Entrez Gene ID
        |  "Gene::" + entrez
        v
  Hetionet Gene::Entrez
```

Final join key: **`hetionet_gene_id`**. `gene_symbol` is annotation only.

Mapping table columns: `ensembl_id`, `gene_symbol`, `entrez_id`,
`hetionet_gene_id`, `hetionet_gene_symbol`, `mapping_status`.

In [ ]:
issues = []

if kirc_source["ensembl_ids"] is not None:
    std = gene_mapping.standardize_gene_ids(kirc_source["ensembl_ids"])
    mapping = std["mapping"]
    kirc_gene_ids = sorted(set(std["usable"]["hetionet_gene_id"].dropna()))
else:
    print("[warn] KIRC expression matrix unavailable -> using the prior-run "
          "Hetionet-side coverage.")
    print("       The Ensembl -> Entrez leg is NOT exercised on this path.")
    mapping = gene_mapping.mapping_from_hetionet_gene_nodes(
        kirc_source["hetionet_gene_ids"]
    )
    kirc_gene_ids = sorted(set(mapping["hetionet_gene_id"].dropna()))
    issues.append({
        "what": "KIRC expression matrix not reachable; gene coverage taken from a prior run",
        "where": "notebook step 5-6",
        "cause": "expression matrix not mounted / not downloadable in this environment",
        "status": "open - re-run with the matrix mounted for real Ensembl counters",
    })

print(f"\nKIRC-mapped Hetionet genes: {len(kirc_gene_ids):,}")
mapping.head()

## 7. Mapping validation

Reports `KIRC genes / HGNC mapped / Entrez mapped / Hetionet mapped / Unmapped /
Duplicated / Final usable genes`, and checks duplicate Ensembl IDs, one-to-many
Ensembl<->Entrez, Entrez IDs missing from Hetionet, and the final intersection.

Nothing is auto-corrected: anomalies are printed and written into the README.

In [ ]:
validation = gene_mapping.validate_gene_mapping(
    mapping,
    kirc_ensembl_ids=kirc_source["ensembl_ids"],
    het_genes=het_genes,
    n_kirc_genes=(None if kirc_source["ensembl_ids"] is not None
                  else len(kirc_source["hetionet_gene_ids"])),
)

for issue in validation["issues"]:
    issues.append({
        "what": issue,
        "where": "gene_mapping.validate_gene_mapping()",
        "cause": "Ensembl/Entrez identifier systems are not 1:1",
        "status": "recorded, not auto-corrected",
    })

## 8. Context configuration

One switch, two contexts - and one shared implementation:

```python
GRAPH_CONTEXT = "pathway"          # or "biological_process"

CONTEXT_CONFIG = {
    "pathway":            {"node_kind": "Pathway",            "metaedge": "GpPW"},
    "biological_process": {"node_kind": "Biological Process", "metaedge": "GpBP"},
}


def get_context_nodes(nodes, graph_context):
    config = CONTEXT_CONFIG[graph_context]
    return nodes[nodes["kind"] == config["node_kind"]].copy()


def get_context_edges(edges, graph_context):
    config = CONTEXT_CONFIG[graph_context]
    return edges[edges["metaedge"] == config["metaedge"]].copy()
```

```text
Gene --GpPW--> Pathway
Gene --GpBP--> Biological Process
```

Both produce the same schema: `gene_id`, `gene_symbol`, `context_id`,
`context_name`, `edge_type`, `context_type`.

In [ ]:
from config import GRAPH_CONTEXT, CONTEXT_CONFIG
from kirc_hetionet.context import (
    get_context_nodes,
    get_context_edges,
    run_context_experiment,
    run_all_context_experiments,
)

print("GRAPH_CONTEXT =", repr(GRAPH_CONTEXT))
for name, cfg in CONTEXT_CONFIG.items():
    print(f"  {name:<20} node_kind={cfg['node_kind']!r:<22} metaedge={cfg['metaedge']!r}")

# the two accessors, on the currently selected context
print()
print("context nodes:", get_context_nodes(nodes, GRAPH_CONTEXT).shape)
print("context edges:", get_context_edges(edges, GRAPH_CONTEXT).shape)

## 9. Pathway / Biological Process pipeline

Single context - change `GRAPH_CONTEXT` in `config.py` (or pass it here) and the
whole thing runs in the other context.

In [ ]:
single = run_context_experiment(GRAPH_CONTEXT, kirc_gene_ids=kirc_gene_ids,
                                nodes=nodes, edges=edges)
single["gene_context_edges"].head()

Both contexts at once - this is the only call that is needed:

In [ ]:
results = run_all_context_experiments(kirc_gene_ids=kirc_gene_ids,
                                      nodes=nodes, edges=edges)

## 10. Result comparison

Per-context outputs never overwrite each other:

```text
results/
├── pathway/{context_nodes,gene_context_edges,subgraph_nodes}.tsv
├── biological_process/{context_nodes,gene_context_edges,subgraph_nodes}.tsv
└── comparison/context_comparison.tsv
```

In [ ]:
comparison = results["comparison"]
print(context.format_comparison(comparison))
comparison

## 11. README update

In [ ]:
from kirc_hetionet import report  # the same writer the headless runner uses

report.update_readme(
    results,
    comparison,
    {"validation": validation},
    kirc_source,
    "ensembl -> hgnc -> entrez -> hetionet"
    if kirc_source["ensembl_ids"] is not None else "prior run (Hetionet side only)",
    issues,
)
print("README updated ->", config.README_PATH)
print(readme_log.get_section("Current Status")[:400])

## 12. Google Drive backup

`save_project_to_drive()` mirrors code, notebook, README, `data/processed`,
`data/external` and `results` into `DRIVE_PROJECT_DIR`, then
`verify_drive_backup()` re-checks the files **on disk**. A file that could not
be written is reported as `[FAIL]`, never as `[OK]`.

The running `.ipynb` is not a file in Colab, so it is snapshotted from the
frontend via `get_ipynb`; if that is unavailable the repository copy is used and
the method actually used is printed.

In [ ]:
report = drive_sync.save_project_to_drive()
print()
status = drive_sync.verify_drive_backup()
print()
print("all verified:", status["all_ok"])